# Calendar Delta Sync
Syncs calendar events from a shared mailbox via the Microsoft Graph `calendarView/delta` endpoint into a Delta Lake table, using incremental delta tokens to only fetch changes on subsequent runs.

In [1]:
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")
spark.conf.set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")

StatementMeta(, e4ebee09-6f0e-4e3e-aa56-fecc60f5d5f2, 3, Finished, Available, Finished, False)

In [ ]:
import requests, time, json
from datetime import datetime, timezone
import pandas as pd
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import lit, current_timestamp

# ── Config ─────────────────────────────────────────────
MAILBOX = "<SHARED_MAILBOX>"
TABLE_NAME    = "CalendarEvents"
TOKEN_TABLE   = "CalendarSyncTokens"     # stores the deltaLink between runs
MAX_RETRIES   = 5

StatementMeta(, e4ebee09-6f0e-4e3e-aa56-fecc60f5d5f2, 4, Finished, Available, Finished, False)

## Authenticating

In [ ]:
tenant_id = "<TENANT_ID>"
client_id = "<CLIENT_ID>"
client_secret = "<CLIENT_SECRET>"

session = requests.Session()
token_resp = session.post(
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token",
    data={
        "client_id":     client_id,
        "client_secret": client_secret,
        "scope":         "https://graph.microsoft.com/.default",
        "grant_type":    "client_credentials",
    },
    timeout=30,
).json()

assert "access_token" in token_resp, f"Auth failed: {token_resp}"
session.headers.update({
    "Authorization": f"Bearer {token_resp['access_token']}",
    "Prefer":        'outlook.timezone="UTC"',
    "Accept":        "application/json",
})
print("Authenticated ✓")

StatementMeta(, e4ebee09-6f0e-4e3e-aa56-fecc60f5d5f2, 5, Finished, Available, Finished, False)

Authenticated ✓


## Loading previous delta token

In [4]:
delta_link = None
if spark.catalog.tableExists(TOKEN_TABLE):
    rows = spark.table(TOKEN_TABLE).collect()
    if rows:
        delta_link = rows[0]["delta_link"]
        print(f"Resuming from saved deltaLink (token length: {len(delta_link)})")
else:
    print("No previous sync token found — will do initial full sync")

# delta_link = None  # ← TEMP: force fresh sync to clear stale delta token

StatementMeta(, e4ebee09-6f0e-4e3e-aa56-fecc60f5d5f2, 6, Finished, Available, Finished, False)

Resuming from saved deltaLink (token length: 365)


## Fetching changes via delta query
First run hits `/calendarView/delta` with a date window for a full sync. Subsequent runs use the saved `deltaLink` to only fetch changes.

> **Note:** `$select` and `$top` are intentionally omitted — the `calendarView/delta` endpoint returns HTTP 400 when these parameters are included.

In [ ]:
if delta_link:
    url = delta_link
    params = None
else:
    url = f"https://graph.microsoft.com/v1.0/users/{MAILBOX}/calendarView/delta"
    params = {
        "startDateTime": "2026-01-01T00:00:00Z",
        "endDateTime":   "2026-12-31T23:59:59Z",
    }

upserts     = []   # changed or new events
deleted_ids = []   # events that were removed
page = 0
new_delta_link = None

t0 = time.time()
print(f"delta_link value going into sync: {delta_link}")
while url:
    retries = 0
    while True:
        try:
            r = session.get(url, params=params, timeout=120)
        except requests.exceptions.Timeout:
            retries += 1
            if retries > MAX_RETRIES:
                raise RuntimeError(f"Page {page+1}: exceeded {MAX_RETRIES} retries on timeout")
            print(f"  page {page+1} timed out (attempt {retries}/{MAX_RETRIES}), retrying…")
            time.sleep(5 * retries)
            continue

        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", "10"))
            print(f"  throttled → waiting {wait}s")
            time.sleep(wait)
            continue
        if r.status_code in (502, 503, 504):
            retries += 1
            if retries > MAX_RETRIES:
                raise RuntimeError(f"Page {page+1}: exceeded {MAX_RETRIES} retries on {r.status_code}")
            wait = 10 * retries
            print(f"  {r.status_code} on page {page+1} (attempt {retries}/{MAX_RETRIES}), retrying in {wait}s…")
            time.sleep(wait)
            continue        

        if not r.ok:
            print(f"HTTP error {r.status_code} on page {page+1}")
            try:
                print("Response body:", r.json())
            except Exception:
                print("Response text:", r.text[:2000])
            r.raise_for_status()

        break

    body = r.json()

    for ev in body.get("value", []):
        if "@removed" in ev:
            deleted_ids.append(ev["id"])
        else:
            upserts.append(ev)

    url = body.get("@odata.nextLink")
    if not url:
        new_delta_link = body.get("@odata.deltaLink")
    params = None   # nextLink already includes query params
    page += 1
    print(f"  page {page}: {len(upserts):,} upserts, {len(deleted_ids):,} deletes so far")

elapsed = time.time() - t0
print(f"\nDelta sync complete: {len(upserts):,} upserts, {len(deleted_ids):,} deletes "
      f"in {page} pages ({elapsed:.1f}s)")

## Flatten upsert events into a DataFrame

In [6]:
FLAT_MAP = {
    "id": "id", "subject": "subject", "bodyPreview": "bodyPreview",
    "start.dateTime": "StartDateTime", "start.timeZone": "StartTZ",
    "end.dateTime": "EndDateTime",     "end.timeZone": "EndTZ",
    "location.displayName": "Location",
    "isAllDay": "isAllDay", "isCancelled": "isCancelled",
    "showAs": "showAs", "sensitivity": "sensitivity",
    "importance": "importance",
    "type": "type", "webLink": "webLink",
    "createdDateTime": "createdDateTime",
    "lastModifiedDateTime": "lastModifiedDateTime",
}

if upserts:
    pdf = pd.json_normalize(upserts)

    # Handle categories before renaming/filtering columns
    if "categories" in pdf.columns:
        pdf["categories"] = [
            "; ".join(c) if isinstance(c, list) else ""
            for c in pdf["categories"]
        ]
    else:
        pdf["categories"] = ""

    # Keep only the columns we need and rename
    available = [c for c in FLAT_MAP.keys() if c in pdf.columns]
    cats = pdf["categories"]
    pdf = pdf[available].rename(columns=FLAT_MAP)
    pdf["categories"] = cats.values

    # Convert datetime columns
    dt_cols = [c for c in ["StartDateTime", "EndDateTime", "createdDateTime", "lastModifiedDateTime"] if c in pdf.columns]
    pdf[dt_cols] = pdf[dt_cols].apply(pd.to_datetime, errors="coerce")

    # Sync metadata
    pdf["is_deleted"] = False
    pdf["sync_timestamp"] = datetime.now(timezone.utc)

    print(f"Upsert DataFrame: {pdf.shape[0]:,} rows × {pdf.shape[1]} cols")
else:
    pdf = pd.DataFrame()
    print("No upserts in this sync")

StatementMeta(, e4ebee09-6f0e-4e3e-aa56-fecc60f5d5f2, 8, Finished, Available, Finished, False)

Upsert DataFrame: 17 rows × 20 cols


## Merge upserts and soft-delete removed events

In [7]:
# ── One-off schema migration (safe to remove after first successful run) ──
if spark.catalog.tableExists(TABLE_NAME):
    existing_cols = [f.name for f in spark.table(TABLE_NAME).schema.fields]
    if "is_deleted" not in existing_cols:
        spark.sql(f"ALTER TABLE {TABLE_NAME} ADD COLUMNS (is_deleted BOOLEAN, sync_timestamp TIMESTAMP)")
        spark.sql(f"UPDATE {TABLE_NAME} SET is_deleted = false, sync_timestamp = current_timestamp() WHERE is_deleted IS NULL")
        print("Schema migrated ✓")

StatementMeta(, e4ebee09-6f0e-4e3e-aa56-fecc60f5d5f2, 9, Finished, Available, Finished, False)

In [8]:
if spark.catalog.tableExists(TABLE_NAME):
    tgt = DeltaTable.forName(spark, TABLE_NAME)

    # 1) Upsert changed/new events
    if not pdf.empty:
        sdf = spark.createDataFrame(pdf)
        if "is_deleted" not in sdf.columns:
            sdf = sdf.withColumn("is_deleted", lit(False))
        if "sync_timestamp" not in sdf.columns:
            sdf = sdf.withColumn("sync_timestamp", current_timestamp())
        if "categories" in tgt.toDF().columns and "categories" not in sdf.columns:
            sdf = sdf.withColumn("categories", lit(""))

        (tgt.alias("t")
            .merge(sdf.alias("s"), "t.id = s.id")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())
        print(f"Merged {len(pdf):,} upserts")

    # 2) Soft-delete removed events
    if deleted_ids:
        del_sdf = spark.createDataFrame(
            [(did,) for did in deleted_ids],
            schema=StructType([StructField("id", StringType())])
        )
        (tgt.alias("t")
            .merge(del_sdf.alias("d"), "t.id = d.id")
            .whenMatchedUpdate(set={
                "is_deleted":     lit(True),
                "sync_timestamp": current_timestamp(),
            })
            .execute())
        print(f"Soft-deleted {len(deleted_ids):,} events")

    if pdf.empty and not deleted_ids:
        print("No changes since last sync ✓")

else:
    if not pdf.empty:
        sdf = spark.createDataFrame(pdf)
        sdf.write.saveAsTable(TABLE_NAME)
        print(f"Created {TABLE_NAME} with {len(pdf):,} rows")
    else:
        print("No events found on initial sync")

StatementMeta(, e4ebee09-6f0e-4e3e-aa56-fecc60f5d5f2, 10, Finished, Available, Finished, False)

Merged 17 upserts
Soft-deleted 2 events


## Persist delta token for next run

In [10]:
if new_delta_link:
    token_sdf = spark.createDataFrame(
        [(MAILBOX, new_delta_link, datetime.now(timezone.utc).isoformat())],
        schema=StructType([
            StructField("mailbox", StringType()),
            StructField("delta_link", StringType()),
            StructField("saved_at", StringType()),
        ])
    )
    token_sdf.write.mode("overwrite").saveAsTable(TOKEN_TABLE)
    print(f"Saved deltaLink to {TOKEN_TABLE} (length: {len(new_delta_link)})")
else:
    print("WARNING: No deltaLink returned — next run will do a full sync")

StatementMeta(, e4ebee09-6f0e-4e3e-aa56-fecc60f5d5f2, 12, Finished, Available, Finished, False)

Saved deltaLink to CalendarSyncTokens (length: 365)


## Summary

In [ ]:
if spark.catalog.tableExists(TABLE_NAME):
    df_table = spark.table(TABLE_NAME)
    total = df_table.count()

    if "is_deleted" in df_table.columns:
        active = df_table.filter("is_deleted = false").count()
        deleted = total - active
        print(f"Table {TABLE_NAME}: {total:,} total rows ({active:,} active, {deleted:,} soft-deleted)")
    else:
        print(f"Table {TABLE_NAME}: {total:,} total rows")